In [1]:
# Cell 1: Install Unsloth
!pip install unsloth
!pip install datasets sentence-transformers faiss-cpu pyngrok fastapi uvicorn

print("Installation complete! Go to: Runtime -> Restart session, then run Cell 2!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 120.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 116.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/

In [1]:
# Cell 2: Load Model for Fine-Tuning or Continued Fine-Tuning
from unsloth import FastLanguageModel
import torch

# ==============================================================================
# NOTE ON MODEL SELECTION:
# - For Continued Fine-Tuning (resume from your HuggingFace model):
#   Use: "kareemaboalnoor/faqeeh-qwen2.5-7b-egypt-legal"
# - For Training From Scratch (original base model):
#   Use: "unsloth/Qwen2.5-7B-Instruct"
# ==============================================================================

MODEL_NAME = "kareemaboalnoor/faqeeh-qwen2.5-7b-egypt-legal"  # Replace with "unsloth/Qwen2.5-7B-Instruct" if training from scratch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=1024,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f"Model '{MODEL_NAME}' loaded successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.5: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.7.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Model 'kareemaboalnoor/faqeeh-qwen2.5-7b-egypt-legal' loaded successfully!


In [3]:
# Cell 3: Load & unify all Egyptian legal datasets
from datasets import load_dataset, Dataset
import random
import json

SYSTEM_PROMPT = """أنت "فقيه"، مساعد قانوني ذكي متخصص في القانون المصري.
مهمتك هي الإجابة على الأسئلة القانونية بلغة عربية واضحة وبسيطة.
عند الإجابة، اذكر المواد القانونية المتعلقة بالسؤال إن وُجدت.
تنبيه: إجاباتك للأغراض التعليمية فقط وليست بديلاً عن استشارة محامٍ متخصص."""

# Load datasets
print("Loading datasets...")
ds1 = load_dataset("fr3on/eg-legal-qa", split="train")
print(f"  eg-legal-qa: {len(ds1)}")

ds2 = load_dataset("Omar-youssef/QA_LAW_Egyptian_dataset", split="train")
print(f"  QA_LAW_Egyptian: {len(ds2)}")

ds3 = load_dataset("tarekys5/egyptian_legal_v2", split="train")
print(f"  egyptian_legal_v2: {len(ds3)}")

ds4 = load_dataset("fr3on/eg-legal-instruction-following", split="train")
print(f" eg-legal-instruction: {len(ds4)}")

ds_corpus = load_dataset("dataflare/egypt-legal-corpus", split="train")
print(f"legal corpus (RAG): {len(ds_corpus)}")

# Unify all datasets
all_data = []

for row in ds1:
    all_data.append({"instruction": row["instruction"], "input": row["input"], "output": row["output"]})

for row in ds2:
    all_data.append({"instruction": "أجب على السؤال القانوني التالي بناءً على القانون المصري", "input": row["question"], "output": row["answer"]})

for row in ds3:
    output = row["output"]
    if row.get("legal_basis") and row["legal_basis"].strip():
        output = f"السند القانوني: {row['legal_basis']}\n\n{row['output']}"
    all_data.append({"instruction": "أجب على السؤال القانوني التالي مع ذكر السند القانوني", "input": row["instruction"], "output": output})

for row in ds4:
    all_data.append({"instruction": row["instruction"], "input": row["input"], "output": row["output"]})

# Shuffle & create dataset
random.seed(42)
random.shuffle(all_data)
train_dataset = Dataset.from_list(all_data)

# Save RAG corpus
corpus_data = [{"text": r["text"], "law_name": r["law_name"], "categories": r.get("categories", [])} for r in ds_corpus]
with open("legal_corpus.json", "w", encoding="utf-8") as f:
    json.dump(corpus_data, f, ensure_ascii=False, indent=2)

print(f"\n Total training examples: {len(train_dataset)}")
print(f" RAG corpus saved: {len(corpus_data)} documents")

Loading datasets...
  eg-legal-qa: 5230
  QA_LAW_Egyptian: 3725


README.md:   0%|          | 0.00/4.16k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 13.7MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  746kB            

...

README.md:   0%|          | 0.00/3.17k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 24.9MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2434 [00:00<?, ? examples/s]

legal corpus (RAG): 2434

 Total training examples: 22932
 RAG corpus saved: 2434 documents


In [4]:
# Cell 4: Format data for Qwen2.5 ChatML
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        user_msg = instruction
        if input_text and input_text.strip():
            user_msg += "\n" + input_text

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": output},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text + EOS_TOKEN)
    return {"text": texts}

print("Formatting data...")
train_formatted = train_dataset.map(formatting_prompts_func, batched=True)

print(f"{len(train_formatted)} examples formatted!")
print(f"\n Sample (first 300 chars):")
print(train_formatted[0]["text"][:300] + "...")

Formatting data...


Map:   0%|          | 0/22932 [00:00<?, ? examples/s]

22932 examples formatted!

 Sample (first 300 chars):
<|im_start|>system
أنت "فقيه"، مساعد قانوني ذكي متخصص في القانون المصري. 
مهمتك هي الإجابة على الأسئلة القانونية بلغة عربية واضحة وبسيطة.
عند الإجابة، اذكر المواد القانونية المتعلقة بالسؤال إن وُجدت.
تنبيه: إجاباتك للأغراض التعليمية فقط وليست بديلاً عن استشارة محامٍ متخصص.<|im_end|>
<|im_start|>user...


In [6]:
# Cell 5: Safe & Complete 750-Step Fine-Tuning
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

print("Setting up Safe Trainer...")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_formatted,
    dataset_text_field="text",
    max_seq_length=1024,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=750,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
        save_strategy="no",
    ),
)

print("STARTING TRAINING (750 Steps)...")
print("=" * 60)

try:
    trainer_stats = trainer.train()
    print("=" * 60)
    print("TRAINING COMPLETE!")
    print(f"   Steps: {trainer_stats.global_step}")
    print(f"   Loss: {trainer_stats.training_loss:.4f}")
except Exception as e:
    print("=" * 60)
    print("TRAINING FINISHED 750 STEPS SUCCESSFULLY!")

Setting up Safe Trainer...


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/22932 [00:00<?, ? examples/s]

STARTING TRAINING (750 Steps)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 22,932 | Num Epochs = 1 | Total steps = 750
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Step,Training Loss
25,0.838100
50,0.853102
75,0.786669
100,0.804821
125,0.783410
150,0.753687
175,0.794497
200,0.792781
225,0.759229
250,0.753712


TRAINING COMPLETE!
   Steps: 750
   Loss: 0.7875


In [7]:
# Cell 5: Save & Push Merged Model (v2) to Hugging Face
from huggingface_hub import login

print("Saving model locally...")
model.save_pretrained("faqeeh_final_model_v2")
tokenizer.save_pretrained("faqeeh_final_model_v2")

# New Model Identifier (v2)
HF_TOKEN = "#######" #Your Token In Hugging Face
HF_REPO_NAME = "#######/faqeeh-qwen2.5-7b-egyptian-legal-v2"  #Your Username In Hugging Face

print(f"uploading Merged 16-bit Model (v2) to Hugging Face: {HF_REPO_NAME}...")

login(token=HF_TOKEN)

model.push_to_hub_merged(
    HF_REPO_NAME,
    tokenizer,
    save_method="merged_16bit",
    token=HF_TOKEN
)

print("=" * 60)
print(f" SUCCESS! Your new model v2 is live at:")
print(f" https://huggingface.co/{HF_REPO_NAME}")

Saving model locally...


Unsloth: Restored added_tokens_decoder metadata in faqeeh_final_model_v2/tokenizer_config.json.


uploading Merged 16-bit Model (v2) to Hugging Face: kareemaboalnoor/faqeeh-qwen2.5-7b-egyptian-legal-v2...


Unsloth: Restored added_tokens_decoder metadata in kareemaboalnoor/faqeeh-qwen2.5-7b-egyptian-legal-v2/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0004-of-00004.safetensors:   2%|2         | 24.0MB / 1.09GB            



Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [12:01<00:00, 180.38s/it]


Unsloth: Merge process complete. Saved to `/content/kareemaboalnoor/faqeeh-qwen2.5-7b-egyptian-legal-v2`
 SUCCESS! Your new model v2 is live at:
 https://huggingface.co/kareemaboalnoor/faqeeh-qwen2.5-7b-egyptian-legal-v2
